In [ ]:
import os
import copy
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.utils import dropout_edge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from torch_geometric.loader import DataLoader

# Paths
DATASET_PATH = "your path"
RESULT_PATH = "your path" 
""
os.makedirs(RESULT_PATH, exist_ok=True)

# Load dataset
df = pd.read_csv(DATASET_PATH)
df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y')
df = df.sort_values(['Company', 'Date']).reset_index(drop=True)

# Normalize features
features = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
scalers = {}
for feat in features:
    scaler = StandardScaler()
    df[feat] = scaler.fit_transform(df[[feat]])
    scalers[feat] = scaler

# Define prediction target: next day Close price
df['Target_Close'] = df.groupby('Company')['Close'].shift(-1)
df = df.dropna(subset=['Target_Close']).reset_index(drop=True)

# Create graphs per company per month
def create_graphs(df):
    graphs = []
    df['YearMonth'] = df['Date'].dt.to_period('M')
    grouped = df.groupby(['Company', 'YearMonth'])

    for (company, ym), group in grouped:
        group = group.sort_values('Date').reset_index(drop=True)
        if len(group) < 2:
            continue
        x = torch.tensor(group[features].values, dtype=torch.float)
        y = torch.tensor([group['Target_Close'].values[-1]], dtype=torch.float).view(-1, 1)

        edge_index = []
        for i in range(len(group) - 1):
            edge_index.append([i, i+1])
            edge_index.append([i+1, i])
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

        data = Data(x=x, edge_index=edge_index, y=y)
        data.company = company
        data.year_month = str(ym)
        graphs.append(data)
    return graphs

graphs = create_graphs(df)
print(f"Total graphs created: {len(graphs)}")

# Split graphs by company (simulate clients)
clients = {}
for g in graphs:
    clients.setdefault(g.company, []).append(g)
print(f"Clients: {list(clients.keys())}")

# Model with contrastive learning support
class GCNContrastive(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(GCNContrastive, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.lin = torch.nn.Linear(hidden_channels, 1)

    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

    def forward(self, x, edge_index, batch):
        node_emb = self.encode(x, edge_index)
        graph_emb = global_mean_pool(node_emb, batch)
        out = self.lin(graph_emb)
        return out, graph_emb

# Graph augmentation: edge dropout
def graph_augmentation(data, drop_edge_prob=0.2):
    edge_index, _ = dropout_edge(data.edge_index, p=drop_edge_prob)
    data_aug = Data(x=data.x, edge_index=edge_index, y=data.y)
    if hasattr(data, 'batch'):
        data_aug.batch = data.batch
    else:
        data_aug.batch = torch.zeros(data.x.size(0), dtype=torch.long)
    return data_aug

# Contrastive loss (InfoNCE)
def contrastive_loss(z1, z2, temperature=0.5):
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    batch_size = z1.size(0)

    representations = torch.cat([z1, z2], dim=0)
    similarity_matrix = torch.matmul(representations, representations.t())

    mask = (~torch.eye(2 * batch_size, 2 * batch_size, dtype=bool)).float().to(z1.device)
    positives = torch.cat([torch.diag(similarity_matrix, batch_size),
                           torch.diag(similarity_matrix, -batch_size)], dim=0)
    nominator = torch.exp(positives / temperature)
    denominator = mask * torch.exp(similarity_matrix / temperature)
    loss = -torch.log(nominator / denominator.sum(dim=1))
    return loss.mean()

# Contrastive training
def train_contrastive(model, loader, optimizer, device, contrastive_weight=0.1):
    model.train()
    total_loss = 0
    for data in loader:
        data = data.to(device)
        data1 = graph_augmentation(data)
        data2 = graph_augmentation(data)
        data1 = data1.to(device)
        data2 = data2.to(device)

        optimizer.zero_grad()
        out1, z1 = model(data1.x, data1.edge_index, data1.batch.to(device))
        out2, z2 = model(data2.x, data2.edge_index, data2.batch.to(device))

        mse_loss = (F.mse_loss(out1, data.y) + F.mse_loss(out2, data.y)) / 2
        c_loss = contrastive_loss(z1, z2)
        loss = mse_loss + contrastive_weight * c_loss

        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
    return total_loss / len(loader.dataset)

# Evaluation
def evaluate(model, loader, device):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out, _ = model(data.x, data.edge_index, data.batch.to(device))
            preds.append(out.cpu())
            trues.append(data.y.cpu())
    preds = torch.cat(preds).view(-1)
    trues = torch.cat(trues).view(-1)
    mse = mean_squared_error(trues, preds)
    return mse, preds, trues

# Prepare loaders per client
client_loaders = {}
for c, g_list in clients.items():
    n = len(g_list)
    train_n = int(0.8 * n)
    train_loader = DataLoader(g_list[:train_n], batch_size=4, shuffle=True)
    test_loader = DataLoader(g_list[train_n:], batch_size=4, shuffle=False)
    client_loaders[c] = (train_loader, test_loader)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
in_channels = len(features)
hidden_channels = 64

def get_model():
    model = GCNContrastive(in_channels, hidden_channels)
    return model.to(device)

# -------- Nash Bargaining functions ---------

def eval_mse(model, loader, device):
    return evaluate(model, loader, device)[0]

def weight_l2_norm(state_dict_a, state_dict_b):
    s = 0.0
    for k in state_dict_a.keys():
        da = state_dict_a[k]
        db = state_dict_b[k]
        s += torch.norm((da - db).float(), p=2)**2
    return torch.sqrt(s).item()

def nash_bargaining_weights(global_model, local_models, client_loaders, device,
                            alpha=1.0, beta=0.05, tau=0.5, prev_util=None):
    if prev_util is None:
        prev_util = {}

    gW = copy.deepcopy(global_model.state_dict())
    weights = {}
    utilities = {}
    raw_scores = []

    base_mse = {}
    for cname, (_, test_loader) in client_loaders.items():
        base_mse[cname] = eval_mse(global_model, test_loader, device)

    for (cname, (train_loader, test_loader)), lmodel in zip(client_loaders.items(), local_models):
        local_mse = eval_mse(lmodel, test_loader, device)
        improvement = base_mse[cname] - local_mse
        drift = weight_l2_norm(lmodel.state_dict(), gW)

        Ui = alpha * improvement - beta * drift
        di = prev_util.get(cname, 0.0)
        surplus = torch.nn.functional.softplus(torch.tensor((Ui - di)/tau)).item()

        utilities[cname] = Ui
        raw_scores.append((cname, surplus))

    total_surplus = sum(s for _, s in raw_scores)
    if total_surplus <= 1e-12:
        vol = {c: len(train_loader.dataset) for c,(train_loader,_) in client_loaders.items()}
        s = sum(vol.values())
        for c in vol:
            weights[c] = vol[c] / s
        return weights, utilities

    for cname, s in raw_scores:
        weights[cname] = s / total_surplus

    return weights, utilities

def weighted_average(local_state_dicts, client_names, agg_weights):
    keys = local_state_dicts[0].keys()
    out = {k: torch.zeros_like(local_state_dicts[0][k]) for k in keys}
    for sd, cname in zip(local_state_dicts, client_names):
        w = agg_weights[cname]
        for k in keys:
            out[k] += sd[k] * w
    return out

# ---------------- Training loop ----------------
epochs_per_round = 3
rounds = 10

global_model = get_model()
global_weights = global_model.state_dict()

mse_per_round = []
prev_utility = {}

for r in range(rounds):
    print(f"Round {r+1}/{rounds}")
    local_models = []
    client_names = []

    for cname, (train_loader, _) in client_loaders.items():
        local_model = get_model()
        local_model.load_state_dict(global_weights)
        optimizer = torch.optim.Adam(local_model.parameters(), lr=0.01)
        for e in range(epochs_per_round):
            loss = train_contrastive(local_model, train_loader, optimizer, device, contrastive_weight=0.1)
            print(f"Client {cname} Epoch {e+1} Loss: {loss:.4f}")
        local_models.append(local_model)
        client_names.append(cname)

    agg_w, utilities = nash_bargaining_weights(
        global_model=global_model,
        local_models=local_models,
        client_loaders=client_loaders,
        device=device,
        alpha=1.0,
        beta=0.05,
        tau=0.5,
        prev_util=prev_utility
    )

    print("Aggregation weights (Nash Bargaining):")
    for c in client_names:
        print(f"  {c}: weight={agg_w[c]:.4f}, utility={utilities[c]:.6f}")

    local_weights = [m.state_dict() for m in local_models]
    global_weights = weighted_average(local_weights, client_names, agg_w)
    global_model.load_state_dict(global_weights)

    prev_utility = utilities

    all_test_data = []
    for _, (_, test_loader) in client_loaders.items():
        all_test_data += test_loader.dataset
    all_test_loader = DataLoader(all_test_data, batch_size=8, shuffle=False)
    mse, preds, trues = evaluate(global_model, all_test_loader, device)
    print(f"Global Model MSE after round {r+1}: {mse:.6f}")
    mse_per_round.append(mse)

# Save results
plt.figure()
plt.plot(range(1, rounds+1), mse_per_round, marker='o')
plt.title("Global MSE per FL Round with Nash Bargaining Aggregation")
plt.xlabel("Round")
plt.ylabel("MSE")
plt.grid(True)
plt.savefig(os.path.join(RESULT_PATH, "mse_per_round_nash_bargaining.png"))
plt.close()

with open(os.path.join(RESULT_PATH, "mse_per_round_nash_bargaining.txt"), "w") as f:
    for i, val in enumerate(mse_per_round, 1):
        f.write(f"Round {i}: MSE={val}\n")

print(f"Training finished. Results saved in {RESULT_PATH}")


Total graphs created: 2636
Clients: ['DELL', 'IBM', 'INTC', 'MSFT', 'SONY', 'VZ']
Round 1/10
Client DELL Epoch 1 Loss: 0.1522
Client DELL Epoch 2 Loss: 0.1364
Client DELL Epoch 3 Loss: 0.1192
Client IBM Epoch 1 Loss: 0.1282
Client IBM Epoch 2 Loss: 0.1565
Client IBM Epoch 3 Loss: 0.1121
Client INTC Epoch 1 Loss: 0.1432
Client INTC Epoch 2 Loss: 0.1101
Client INTC Epoch 3 Loss: 0.0983
Client MSFT Epoch 1 Loss: 0.1334
Client MSFT Epoch 2 Loss: 0.1138
Client MSFT Epoch 3 Loss: 0.1057
Client SONY Epoch 1 Loss: 0.1722
Client SONY Epoch 2 Loss: 0.1410
Client SONY Epoch 3 Loss: 0.1594
Client VZ Epoch 1 Loss: 0.1098
Client VZ Epoch 2 Loss: 0.1145
Client VZ Epoch 3 Loss: 0.0914
Aggregation weights (Nash Bargaining):
  DELL: weight=0.0619, utility=0.648425
  IBM: weight=0.2134, utility=2.650552
  INTC: weight=0.0185, utility=-0.269584
  MSFT: weight=0.6719, utility=8.354627
  SONY: weight=0.0175, utility=-0.303812
  VZ: weight=0.0170, utility=-0.322228
Global Model MSE after round 1: 0.037651
Ro